# System Kodowania Jednostek Terytorialnych i Statystycznych (KTS)

System Kodowania Jednostek Terytorialnych i Statystycznych (KTS) to przyjęty na potrzeby baz danych i systemów informatycznych statystyki publicznej jednolity wykaz kodów i nazw jednostek zasadniczego trójstopniowego podziału terytorialnego państwa oraz jednostek terytorialnych do celów statystycznych (klasyfikacja NUTS). System KTS wprowadzony został Zarządzeniem wewnętrznym nr 22 Prezesa Głównego Urzędu Statystycznego z dnia 24 sierpnia 2017 r. w sprawie wprowadzenia Systemu Kodowania Jednostek Terytorialnych i Statystycznych. Zastępuje on, na potrzeby statystyki publicznej Nomenklaturę Jednostek Terytorialnych do Celów Statystycznych (NTS), zniesioną z dniem 1 stycznia 2018.

System KTS został opracowany na podstawie krajowego rejestru urzędowego podziału terytorialnego kraju (TERYT) oraz stosowanej w krajach Unii Europejskiej Klasyfikacji Jednostek Terytorialnych do Celów Statystycznych (NUTS).

System KTS dzieli Polskę na terytorialne, hierarchicznie powiązane jednostki na siedmiu poziomach. Każda z jednostek niższego poziomu zawiera się wyłącznie w jednej jednostce poziomu wyższego, przy czym jedna jednostka terytorialna może reprezentować kilka poziomów KTS.

Poziomy terytorialne systemu KTS obejmują:
* Poziom KTS 0 – Polska
* Poziom KTS 1 – Makroregiony (czyli jednostki NUTS 1)
* Poziom KTS 2 – Województwa
* Poziom KTS 3 – Regiony (czyli jednostki NUTS 2)
* Poziom KTS 4 – Podregiony (czyli jednostki NUTS 3)
* Poziom KTS 5 – Powiaty (wraz z miastami na prawach powiatu – analogicznie jak w Rejestrze TERYT)
* Poziom KTS 6 – Gminy (wraz z częściami miejskimi gmin miejsko-wiejskich, częściami wiejskimi gmin miejsko-wiejskich, dzielnicami Warszawy, delegaturami Krakowa, Łodzi, Poznania i Wrocławia – analogicznie jak w Rejestrze TERYT)

[Źródło: System Kodowania Jednostek Terytorialnych i Statystycznych (KTS)](https://statystykaregionalna.new.stat.gov.pl/jednostki-terytorialne/system-kts)

## 0. Installation and Imports of libraries

In [0]:
import re
import pandas as pd
from IPython.display import clear_output
from pyspark.sql.functions import col, trim, last, monotonically_increasing_id, when, upper, broadcast
from pyspark.sql.window import Window

In [0]:
%pip install xlrd
clear_output()

In [0]:
# ── Reusable ANSI formatting codes ─────────────────────────────────────
ft_bold   = "\033[1m"
ft_regular  = "\033[0m"

## 1. Reference to data file

In [0]:
# Path to the file on the volume where we do have extract from KTS system (made by GUS - Główny Urząd Statystyczny)
# FILE_PATH = "/Volumes/workspace/default/simc_gus_gov_pl/kts_2026v2.xls"

In [0]:
# List available volumes in the workspace.default schema
volumes_df = spark.sql("SHOW VOLUMES")

# Assign the first volume name to a variable
volume = volumes_df.select("volume_name").first()[0]
print(f"{ft_bold}Volume name{ft_regular}: {volume}")

In [0]:
# List files in the workspace.default volume whose name contains "kts_", return the path of the most recently modified one.
files_df = spark.read.format("binaryFile") \
    .load(f"/Volumes/workspace/default/{volume}/") \
    .filter(col("path").contains("kts_")) \
    .orderBy(col("modificationTime").desc()) \
    .limit(1)

latest = files_df.select("path", "modificationTime", "length").first()

if latest:
    print(f"{ft_bold}Latest modified file{ft_regular}: {latest["path"]}")
    print(f"{ft_bold}Modified at{ft_regular}:         {latest['modificationTime']}")
    print(f"{ft_bold}Size (bytes){ft_regular}:         {latest['length']}")
else:
    raise FileNotFoundError("No files containing 'kts_' found in the volume.")

FILE_PATH = latest["path"]

In [0]:
# FILE_PATH may start with "dbfs:" or "file:" — normalise for pandas/openpyxl
excel_path = FILE_PATH.replace("dbfs:", "") if FILE_PATH else None

if excel_path:
    xls = pd.ExcelFile(excel_path)
    sheet_names = xls.sheet_names
    print(f"{ft_bold}Available sheets{ft_regular}:", sheet_names)

    # Extract the 4-digit year from sheets matching "KTS ####"
    kts_sheets = []
    for sheet in sheet_names:
        m = re.search(r"KTS\s*(\d{4})", sheet)
        if m:
            kts_sheets.append((sheet, int(m.group(1))))

    if kts_sheets:
        # Sort by year descending and pick the highest
        kts_sheets.sort(key=lambda x: x[1], reverse=True)
        latest_year_sheet = kts_sheets[0][0]
        #print(f"KTS sheets found: {kts_sheets}")
        print(f"{ft_bold}latest_year_sheet{ft_regular} = '{latest_year_sheet}'")
    else:
        latest_year_sheet = None
        print(f'{ft_bold}No sheets matching "KTS ####" were found{ft_regular}.')
else:
    latest_year_sheet = None
    print(f"{ft_bold}FILE_PATH is not set{ft_regular}; cannot list sheets.")

## 2. Apply multi-row schema transformation

In [0]:
HDR_FIRST = 1          # row number of the "KTS 1 / KTS 2 / KTS 3" row
LAST_COL  = "R"        # 6 groups x 3 columns = A:R
LAST_ROW  = 5000     # set above your real last row

def read_range(addr):
    return (spark.read.format("excel")
            .option("headerRows", 0)
            .option("dataAddress", f"{latest_year_sheet}!{addr}")
            .load(FILE_PATH))

# 1) Read the 3 header rows as plain data
hdr = read_range(f"A{HDR_FIRST}:{LAST_COL}{HDR_FIRST + 2}").collect()
group_row, _, leaf_row = [
    [None if v is None else str(v).strip() for v in row] for row in hdr
]

# 2) Forward-fill the merged "KTS n" cells
last = None
for i, v in enumerate(group_row):
    last = v if v else last
    group_row[i] = last

# 3) Map the bilingual leaf headers to your suffixes
def leaf_name(text):
    t = " ".join((text or "").lower().split())      # also collapses the in-cell line break
    if t.startswith("nazwa dodatkowa"): return "CLASS"
    if t.startswith("nazwa"):           return "NAME"
    if t.startswith("kod"):             return "CODE"
    raise ValueError(f"Unmapped header cell: {text!r}")

names = []
for g, l in zip(group_row, leaf_row):
    group = re.sub(r"\s+", "_", g)                  # "KTS 1" -> "KTS_1"
    names.append(f"{group}_{leaf_name(l)}")
# ['KTS_1_CODE', 'KTS_1_Name', 'KTS_1_ADDT_NAME', 'KTS_2_CODE', ...]

# 4) Body starts right after the 3 header rows
df = (read_range(f"A{HDR_FIRST + 3}:{LAST_COL}{LAST_ROW}")
      .toDF(*names)
      .dropna(how="all"))

## 3. Apply gap-filling of missing values

In [0]:
# Fill-down merged cells (forward-fill blanks within logical groups) - Spark doesn't have a built-in fill-down, but you can use windowing:
df = df.withColumn("__row_id__", monotonically_increasing_id())
w = Window.orderBy("__row_id__")

# Apply forward-fill to every column: convert blank strings to NULL, then carry last non-null forward
col_types = {f.name: f.dataType.simpleString() for f in df.schema.fields}
for c in df.columns:
    if c != "__row_id__":
        if col_types[c] == "string":
            df = df.withColumn(c, when(col(c) == "", None).otherwise(col(c)))
        df = df.withColumn(c, last(c, ignorenulls=True).over(w))

df = df.drop("__row_id__").filter(col("KTS_6_CLASS") == "miasto").dropDuplicates()
display(df)

## 4. Enrich with external data

In [0]:
# ── Enrich df with external data ────────────────────────────────────────
# Strategy:
#   1. Scan the volume for enrichment files (area, population, etc.).
#   2. Load matching files and join on city code (KTS_6_CODE) or city name.
#   3. Add a static historical-region mapping (built from common knowledge).
#   4. Provide a template for API-based enrichment (GUS BDL / Nominatim).

# ── 1. Search the volume for enrichment files ────────────────────────────
enrich_files = spark.read.format("binaryFile") \
    .load(f"/Volumes/workspace/default/{volume}/") \
    .filter(
        col("path").contains("area") |
        col("path").contains("population") |
        col("path").contains("ludn") |
        col("path").contains("powierz") |
    ) \
    .orderBy(col("modificationTime").desc())

enrich_rows = enrich_files.select("path", "modificationTime", "length").collect()

if enrich_rows:
    print(f"{ft_bold}Enrichment files found on volume{ft_regular}:")
    for r in enrich_rows:
        print(f"  • {r['path']}  ({r['length']} bytes, modified {r['modificationTime']})")
else:
    print(f"{ft_bold}No enrichment files found{ft_regular} on volume '{volume}'.")
    print("Upload area/population files to the volume or use the API template below.")

# ── 2. Load and join enrichment files (if present) ───────────────────────
enrichment_df = None
tmp = spark.createDataFrame([{}])

for r in enrich_rows:
    fpath = r["path"].replace("dbfs:", "")
    ext = fpath.rsplit(".", 1)[-1].lower()

    if ext in ("xls", "xlsx", "xl"):
        tmp = spark.read.format("excel").option("headerRows", 1).load(fpath)
    elif ext == "csv":
        tmp = spark.read.format("csv").option("header", True).option("inferSchema", True).load(fpath)
    else:
        continue

    print(f"\n{ft_bold}Loaded{ft_regular}: {fpath}")
    print(f"  Columns: {tmp.columns}")
    tmp.printSchema()

    # Normalise join columns — try to find a code or name column
    code_candidates  = [c for c in tmp.columns if "kod" in c.lower() or "code" in c.lower()]
    name_candidates  = [c for c in tmp.columns if "nazwa" in c.lower() or "name" in c.lower()]
    area_candidates   = [c for c in tmp.columns if "powierz" in c.lower() or "area" in c.lower()]
    pop_candidates    = [c for c in tmp.columns if "ludn" in c.lower() or "popul" in c.lower()]

    # Pick best join key
    join_key = None
    if code_candidates:
        join_key = code_candidates[0]
        tmp = tmp.withColumn(join_key, trim(col(join_key)))
        df_join_key = "KTS_6_CODE"
    elif name_candidates:
        join_key = name_candidates[0]
        tmp = tmp.withColumn(join_key, upper(trim(col(join_key))))
        df_join_key = "KTS_6_NAME"

    if join_key:
        # Select only useful columns to avoid column-name collisions
        useful = [join_key] + area_candidates + pop_candidates
        useful = list(dict.fromkeys(useful))  # deduplicate, preserve order
        tmp_sel = tmp.select(*useful)

        # Rename area / population columns for clarity
        rename_map = {}
        for c in area_candidates:
            rename_map[c] = "area_km2"
        for c in pop_candidates:
            rename_map[c] = "population"
        for old, new in rename_map.items():
            if old in tmp_sel.columns and new not in tmp_sel.columns:
                tmp_sel = tmp_sel.withColumnRenamed(old, new)

        if enrichment_df is None:
            enrichment_df = tmp_sel
        else:
            enrichment_df = enrichment_df.join(tmp_sel, on=join_key, how="outer")

if enrichment_df is not None:
    # Join enrichment data onto the main df
    # Determine the join key column name in enrichment_df
    enrich_key = [c for c in enrichment_df.columns if c in code_candidates or c in name_candidates][0]
    df_key = "KTS_6_CODE" if enrich_key in code_candidates else "KTS_6_NAME"

    # Align types / formatting for the join
    if df_key == "KTS_6_NAME":
        enrichment_df = enrichment_df.withColumn(enrich_key, upper(trim(col(enrich_key))))
        df = df.withColumn(df_key, upper(trim(col(df_key))))
    else:
        enrichment_df = enrichment_df.withColumn(enrich_key, trim(col(enrich_key)))
        df = df.withColumn(df_key, trim(col(df_key)))

    df = df.join(broadcast(enrichment_df), df[df_key] == enrichment_df[enrich_key], how="left")
    print(f"\n{ft_bold}Joined enrichment data{ft_regular} on '{df_key}' <-> '{enrich_key}'")

# ── 3. Static historical-region mapping ─────────────────────────────────
# Map Polish voivodeships (KTS_2_CODE) to broad historical regions.
# Codes: 02 dolnośląskie, 04 kujawsko-pomorskie, 06 lubelskie, 08 lubuskie,
#        10 łódzkie, 12 małopolskie, 14 mazowieckie, 16 opolskie, 18 podkarpackie,
#        20 podlaskie, 22 pomorskie, 24 śląskie, 26 świętokrzyskie, 28 warmińsko-mazurskie, 30 wielkopolskie, 32 zachodniopomorskie
historical_region_map = [
    ("02", "Śląsk / Dolny Śląsk"),
    ("04", "Kujawy / Pomorze"),
    ("06", "Lubelszczyzna / Małopolska Wschodnia"),
    ("08", "Lubuskie / Wielkopolska"),
    ("10", "Polska Centralna / Ziemia Łódzka"),
    ("12", "Małopolska"),
    ("14", "Mazowsze"),
    ("16", "Śląsk Opolski"),
    ("18", "Podkarpacie / Galicja"),
    ("20", "Podlasie"),
    ("22", "Pomorze Gdańskie"),
    ("24", "Śląsk / Górny Śląsk"),
    ("26", "Świętokrzyskie / Ziemia Sandomierska"),
    ("28", "Warmia / Mazury / Prusy"),
    ("30", "Wielkopolska"),
    ("32", "Pomorze Zachodnie / Ziemia Lubuska"),
]

hist_region_df = spark.createDataFrame(historical_region_map, ["KTS_2_CODE", "historical_region"])
df = df.join(broadcast(hist_region_df), on="KTS_2_CODE", how="left")

print(f"\n{ft_bold}Added historical_region{ft_regular} column from static voivodeship mapping.")

# ── 4. Template for API-based enrichment (GUS BDL / Nominatim) ────────────
# Uncomment and adapt when ready to fetch live data.

# import requests
#
# # Example: GUS BDL API — fetch population for a given territorial unit
# BDL_API = "https://bdl.stat.gov.pl/api/v1/data/by-variable/{variable_id}?unit-level=6&format=json"
# # variable_id for population: check https://bdl.stat.gov.pl/BDL/dane/indicators
#
# def fetch_population(city_code):
#     resp = requests.get(BDL_API.format(variable_id="<POP_VAR_ID>"),
#                        params={"unit-parent-id": city_code})
#     if resp.ok and resp.json().get("results"):
#         return resp.json()["results"][0]["values"][0]["val"]
#     return None
#
# from pyspark.sql.functions import udf
# from pyspark.sql.types import IntegerType
# pop_udf = udf(fetch_population, IntegerType())
# df = df.withColumn("population_api", pop_udf(col("KTS_6_CODE")))
#
# # Example: Nominatim (OpenStreetMap) — geocode city for lat/lon
# def geocode(city_name):
#     resp = requests.get("https://nominatim.openstreetmap.org/search",
#                         params={"q": f"{city_name}, Polska", "format": "json", "limit": 1},
#                         headers={"User-Agent": "KTS-enrichment/1.0"})
#     if resp.ok and resp.json():
#         return float(resp.json()[0]["lat"]), float(resp.json()[0]["lon"])
#     return None, None
#
# from pyspark.sql.types import StructType, StructField, DoubleType
# geo_schema = StructType([StructField("latitude", DoubleType()), StructField("longitude", DoubleType())])
# geo_udf = udf(geocode, geo_schema)
# df = df.withColumn("coords", geo_udf(col("KTS_6_NAME"))) \
#        .withColumn("latitude", col("coords.latitude")) \
#        .withColumn("longitude", col("coords.longitude")) \
#        .drop("coords")

# ── Show the enriched result ─────────────────────────────────────────────
display(df)
df.printSchema()